# Phase 1 — EDA: Towards Comprehensive Dataset

Loading and merging the training data before building models.
Just checking the shape, class balance, and what the features look like.


## Load and merge

Each row is a message. Merging `Dataset.csv` with `Repetition.csv` on `(User1, User2)` adds repetition stats per message:
- `TotalMsgs`, `AggressiveMsgs` — how active/aggressive this pair has been
- `Peerness` — how peer-like the relationship is (0–1)
- `CBLabel` — target (0 = non-cyberbullying, 1 = cyberbullying)


In [1]:
import sys
from pathlib import Path

# Find F21MP project root (works whether cwd is project root or notebooks/phase1_training/)
ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").is_dir() and (candidate / "requirements.txt").is_file():
        ROOT = candidate
        break
else:
    raise RuntimeError(
        "Cannot find F21MP project root. Open the F21MP folder in Cursor/VS Code, "
        "or run Jupyter with cwd set to the project root."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.phase1_training.data.load_towards_comprehensive import load_and_merge, DEFAULT_MERGED_OUTPUT
import pandas as pd

df = load_and_merge(save_path=DEFAULT_MERGED_OUTPUT)
print(df.shape)
print(df.columns.tolist())
if "CBLabel" in df.columns:
    print(df["CBLabel"].value_counts())
print("Merged data saved to:", DEFAULT_MERGED_OUTPUT)

(15554, 15)
['Date', 'Time', 'User1', 'User2', 'Message', 'SubLabel', 'CBLabel', 'TotalMsgs', 'AggressiveMsgs', 'Peerness', 'age_user1', 'gender_user1', 'age_user2', 'gender_user2', 'age_diff']
CBLabel
0    15428
1      126
Name: count, dtype: int64
Merged data saved to: /Users/sagarmamindla/Desktop/F21MP/outputs/data/merged_towards_comprehensive.csv


## What the data looks like

Dataset is **heavily imbalanced** — about 126 cyberbullying messages out of ~15,500 total (~0.8%).
This means accuracy is a useless metric here; we'll use F1 throughout.

~53 rows drop during training due to missing tabular features after the merge.


In [2]:
# Quick exploration — class balance and sample preview
import pandas as pd

if "CBLabel" in df.columns:
    counts = df["CBLabel"].value_counts()
    pct = (counts / len(df) * 100).round(2)
    print("Class balance:")
    print(pd.DataFrame({"count": counts, "percent": pct}))

print("\nFirst 3 rows (key columns):")
cols = ["User1", "User2", "Message", "CBLabel", "Peerness", "TotalMsgs", "AggressiveMsgs"]
print(df[cols].head(3).to_string())

print("\nPeerness & repetition stats:")
print(df[["Peerness", "TotalMsgs", "AggressiveMsgs"]].describe().round(2).to_string())

Class balance:
         count  percent
CBLabel                
0        15428    99.19
1          126     0.81

First 3 rows (key columns):
   User1  User2                                                                                                                                            Message  CBLabel  Peerness  TotalMsgs  AggressiveMsgs
0    194     89  The last few days I have been working in this article. I think it is now optimal. Could you take a close look at it and rephrase where necessary.        0  0.672222        1.0             0.0
1    139    109                                                           Just another white supremacist dog. 'nuff said.  Only a motherfucker shoots prisoners.          0  0.433333        2.0             1.0
2    175    125                                                                                                                          Support. Same reason as.         0  0.277778        1.0             0.0

Peerness & repetition s

EDA done — merged CSV saved to `outputs/data/merged_towards_comprehensive.csv`.
Run `02_training.ipynb` to train the models.
